# Half-Atwood Machine

Includes 3 methods to analyze the Half-Atwood Machine.

1. Newton's Second Law Approach
2. Work-Energy Approach
3. Energy Conservation Approach

In [1]:
import sympy as sp
import sympy.physics.mechanics as spm
import sympy.physics.vector as spv
from sympy.physics.vector.printing import init_vprinting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import display, Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
init_vprinting(use_latex="mathjax")

HALF = sp.S.Half

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> spm.ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return spm.ReferenceFrame(
        frame,
        latexs=(
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{x}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{y}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{z}}}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> spm.ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

![Atwood machine](..//figures/Half-Atwood-Machine01.jpg)


In [3]:
(R, # Radius of the disc
 m1, # Mass of block 1
 m2, # Mass of block 2
 mP, # Mass of the pulley
 tauf, # Frictional torque on the pulley
 IP, # Moment of inertia of disc about the center of mass
 muk, # Coefficient of kinetic friction
 g # Acceleration due to gravity
) = sp.symbols('R m1 m2 mP tau_f I_P mu_k g', real=True, positive=True)


## Half Atwood Machine using Newton's Laws

In [4]:
display(
    Markdown(
        r"FBD for $m_1$"
        "\n\n"
        r"![FBD for masses](../figures/Half-Atwood-Machine03-m1.jpg)"
    )
)

a = sp.symbols("a", real=True)
A1 = reference_frame("A_1")

N1T = sp.symbols(r"N_{1T}", real=True, positive=True)
T1 = sp.symbols("T_1", real=True, positive=True)
fk = sp.symbols("f_k", real=True, positive=True)

forces_m1 = N1T * A1.y + m1 * g * (-A1.y) + T1 * A1.x + fk * (-A1.x)
display(Math(r"\text{Force on } m_1: " f"{spv.vlatex(forces_m1)}"))

N2Leqn1 = sp.Eq(forces_m1.to_matrix(A1), (m1 * a * A1.x).to_matrix(A1))
display(Math(r"\text{N2L for } m_1: " f"{spv.vlatex(N2Leqn1)}"))

FBD for $m_1$

![FBD for masses](../figures/Half-Atwood-Machine03-m1.jpg)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [5]:
display(
    Markdown(
        r"Pulley kinetics :"
        "\n\n"
        r"![FBD for masses](../figures/Half-Atwood-Machine04-pulley.jpg)"
    )
)

pulley = reference_frame("P")
alpha = sp.symbols(r"alpha", real=True, positive=True)
T2 = sp.symbols("T_2", real=True, positive=True)

vec_alpha = (alpha * pulley.z).subs(alpha, a / R)
tau_p = (
    (R * (-pulley.y)).cross(T1 * (-pulley.x))
    + (R * (pulley.x)).cross(T2 * (pulley.y))
    + tauf * (-pulley.z)
)
display(Math(r"\text{Torque on pulley: }" + spv.vlatex(tau_p)))

N2LR = sp.Eq(tau_p.to_matrix(pulley), (IP * vec_alpha).to_matrix(pulley))

display(Math(r"\text{N2LR for pulley: }" + spv.vlatex(N2LR)))

Pulley kinetics :

![FBD for masses](../figures/Half-Atwood-Machine04-pulley.jpg)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [6]:
display(
    Markdown(
        r"FBD for $m_1$"
        "\n\n"
        r"![FBD for masses](../figures/Half-Atwood-Machine03-m2.jpg)"
    )
)

A2 = reference_frame("A_2")

T2 = sp.symbols("T_2", real=True, positive=True)
fk = sp.symbols("f_k", real=True, positive=True)

forces_m2 = m2 * g * (A2.y) + T2 * (-A2.y) 
display(Math(r"\text{Force on } m_2: " f"{spv.vlatex(forces_m2)}"))

N2Leqn2 = sp.Eq(forces_m2.to_matrix(A2), (m2 * a * A2.y).to_matrix(A2))
display(Math(r"\text{N2L for } m_2: " f"{spv.vlatex(N2Leqn2)}"))

FBD for $m_1$

![FBD for masses](../figures/Half-Atwood-Machine03-m2.jpg)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [7]:
def simplify_value(value):
    num, den = value.as_numer_denom()
    num = (num / R**2).simplify().collect([g])
    den = (den / R**2).simplify()
    return num / den


system_equations = [N2Leqn1, N2Leqn2, N2LR]
unknowns = [T1, T2, N1T, a]
solutions = sp.solve(system_equations, unknowns)

complete_solutions = {
    key: value.subs(fk, muk * N1T).subs(N1T, solutions[N1T])
    for key, value in solutions.items()
}

simplified_solutions = {
    key: sp.simplify(value) for key, value in complete_solutions.items()
}

simplified_complete_solutions = {
    key: simplify_value(value) for key, value in complete_solutions.items()
}

# display the simplified complete solutions on separate lines
for key, value in simplified_complete_solutions.items():
    display(Math(rf"\displaystyle {spv.vlatex(key)} = {spv.vlatex(value)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>